In [1]:
import logging
from exp.run import ExperimentRun, SummarySectionName
from exp.config import TransformerExperiments, CNNExperiments
logging.basicConfig(level=logging.ERROR)

In [2]:
config = TransformerExperiments()
config = CNNExperiments()
config.debug = False
config.repeats = 1
config.gpu_id = 1

exp = ExperimentRun(config=config)

In [3]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"PyTorch built with CUDA version: {torch.version.cuda}")
print(f"CUDA version: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")

PyTorch version: 2.6.0+cu124
PyTorch built with CUDA version: 12.4
CUDA version: True
CUDA device count: 2


In [4]:
model = config.models[-1]
optimizer = config.optimisers[0]
batch = 400
gpu_id = 1
task_id = None
in_docker =True

In [5]:
# for model in models:
#     for batch in range(10, 15, 5):
#         for i in range(config.repeats):
#             exp.add_task(
#                 model_name=model,
#                 batch_size=batch,
#                 optimizer=optimizer,
#                 gpu_id=gpu_id,
#                 task_id=task_id,
#             )
# for model in config.models:
#     for i in range(1):
#         exp.add_task(
#             model_name=model,
#             batch_size=batch,
#             optimizer=optimizer,
#             gpu_id=gpu_id,
#             task_id=task_id,
#         )
# exp.add_task(
#     model_name=model,
#     batch_size=batch,
#     optimizer=optimizer,
#     gpu_id=gpu_id,
#     task_id=task_id,
# )



## Measure Ground Truth and Estimated Memory for Each job

In [6]:
exp.run_group_truth(in_docker=in_docker)

100%|██████████| 560/560 [00:08<00:00, 67.77it/s]


=============== Start massively run for GPU train ======================


100%|██████████| 560/560 [11:16:31<00:00, 72.49s/it]   
0it [00:00, ?it/s]
0it [00:00, ?it/s]


## Estimate Max GPU Memory by DNNmem

In [7]:
est_list = [
    SummarySectionName.DNNmem,
    SummarySectionName.schedtune
]
if not isinstance(exp._config, CNNExperiments):
    est_list.append(SummarySectionName.LLmem)
exp.run_estimation(
    estimators=est_list,
    in_docker=in_docker
)

if isinstance(exp._config, CNNExperiments):
    exp.verify_llmem_result()



================== Create docker containers ==================


100%|██████████| 560/560 [19:03<00:00,  2.04s/it]


================== Execute docker containers ==================
=============== Start massively run for GPU train ======================


100%|██████████| 560/560 [2:35:54<00:00, 16.70s/it]  
0it [00:00, ?it/s]


================== Statistics ==================
Run(success/total): 1120/1120


100%|██████████| 560/560 [00:00<00:00, 85315.11it/s]


## Estimate Max GPU Memory by SchedTune

In [8]:
exp.statistics()
results = exp.to_evaluation_result()

100%|██████████| 583/583 [00:00<00:00, 19674.93it/s]


=============== Statistics for CNN-Exp ==================
train: 583/583
config: 583/583
groundtruth: 560/583
solution: 560/583
schedtune: 560/583
DNNmem: 560/583
LLmem: 0/583


100%|██████████| 583/583 [00:00<00:00, 7667.74it/s]
